# 18 - Persiapan Eksperimen RAG

Setup, konfigurasi, registrasi model/prompt, unduh artefak HF, dan penentuan gold chunk untuk Hit@K/Recall@K/MRR. Acuan: Bab 3 §3.2.4-3.2.5. Lanjutan: notebook 19 (generasi), 20 (retrieval metrics + RAGAS), 21 (fungsi/robustness/timing, lokal).

## 1. Setup paket

In [1]:
import subprocess, sys
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "pandas==2.2.3", "requests>=2.32,<3",
        "transformers==4.46.3", "huggingface-hub==0.27.1",
        "langchain==0.3.14", "langchain-core==0.3.29",
        "sentence-transformers==3.3.1", "einops>=0.7,<1",
        "faiss-cpu==1.9.0.post1", "rank-bm25==0.2.2",
        "langchain-community==0.3.14"])
    print("Selesai. RESTART KERNEL sekarang (Run > Restart & clear outputs), lalu set "
          "INSTALL_DEPENDENCIES=False sebelum menjalankan ulang dari sel ini.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.29 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.29 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.27.1 which is incompatible.
gradio 5.50

Selesai. RESTART KERNEL sekarang (Run > Restart & clear outputs), lalu set INSTALL_DEPENDENCIES=False sebelum menjalankan ulang dari sel ini.


In [2]:
from pathlib import Path
import os, json, hashlib, ast

# hash konten untuk mencegah pencampuran eksperimen
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# simpan JSON atomik
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

# ambil secret Kaggle tanpa mencetak nilainya
def read_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Aktifkan secret {name} pada Kaggle Add-ons > Secrets.")
    return value

HF_TOKEN = read_secret("HF_Faiss")

## 2. Konfigurasi dan pemulihan hasil

In [3]:
# Test/Run total
RUN_NAME = "Penuh"
DATA_SPLIT = "test"      # pilot: val. final: test, RUN_NAME baru
SMOKE_TEST = False       # True: ambil sedikit acak untuk cek kode cepat, abaikan stratifikasi
SMOKE_TEST_N = 5
SAMPLE_FRACTION = 0.05  # dipakai kalau SMOKE_TEST=False - 5% stratified by subtask_type
SEED = 42

# path folder
RESTORE_DIR = ""        # opsional: /kaggle/input/.../pilot_v2
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "rag_results" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
KB_DIR = WORK / "rag_assets" / "kb"

# Hugging face
DATASET_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
HF_KB_REPO = "Makaareeem/publikasi-rag-knowledge-base"
DATASET_REVISION = "main"
KB_REVISION = "main"
EMBEDDING_REVISION = "main"
RERANKER_REVISION = "main"

# conf ollama
INFERENCE_URL = "http://127.0.0.1:11434"
NUM_CTX = 8192

# pulihkan artefak run sebelumnya tanpa menimpa berkas berbeda
def restore_run(source_dir, target_dir):
    import shutil
    src, dst = Path(source_dir).resolve(), Path(target_dir).resolve()
    if src == dst or src in dst.parents or dst in src.parents:
        raise ValueError("Folder sumber dan tujuan restore harus terpisah.")
    files = [p for p in src.rglob("*") if p.is_file()]
    for p in files:
        target = dst / p.relative_to(src)
        if target.exists() and target.read_bytes() != p.read_bytes():
            raise ValueError(f"Konflik restore: {target.name}; pilih folder output kosong.")
    for p in files:
        target = dst / p.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(p, target)
            

if RESTORE_DIR:
    restore_run(RESTORE_DIR, OUT)
assert DATA_SPLIT in ["val", "test"]
assert DATA_SPLIT != "test" or not SMOKE_TEST, "Final (test) tidak boleh SMOKE_TEST."

## 3. Registrasi model dan prompt

In [4]:
# RAG sesuai system
EMBEDDING_MODEL_ID = "nomic-ai/nomic-embed-text-v1.5"
QUERY_PREFIX = "search_query: "
RETRIEVAL_K = 5
TOP_K_DENSE = 20
TOP_K_SPARSE = 20
TOP_K_RERANK_CANDIDATES = 20
RRF_K = 60
MAX_NEW_TOKENS = 512
RERANKER_MODEL_ID = "madebyaris/rerank-indonesia"
RERANK_SCORE_THRESHOLD = -1.0

SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS "
    "berdasarkan konteks yang diberikan. Kutip bagian konteks yang relevan sebelum "
    "menjawab, merujuk sumber HANYA dengan nomor sitasi seperti [1] [2], jangan "
    "menyebut ulang judul dokumen dalam kalimat jawaban karena berisiko salah "
    "mengaitkan dengan sumber lain. HANYA gunakan angka atau fakta yang secara "
    "eksplisit tertulis dalam konteks yang diberikan. JANGAN menambahkan angka dari "
    "pengetahuan Anda sendiri walau terasa familiar atau masuk akal. Jika konteks "
    "memuat TARGET, RENCANA, atau PROYEKSI, jelaskan dengan jelas bahwa itu adalah "
    "target/rencana, BUKAN angka realisasi. Jika konteks tidak memuat informasi yang "
    "relevan, katakan dengan jujur bahwa Anda tidak memiliki informasi tersebut."
)

# K1-K4
MODEL_REGISTRY = {
    "llama3.2-base": {"ollama_name": "llama3.2-base", "gguf_repo": "Makaareeem/llama3.2-base-gguf",
        "gguf_filename": "Llama-3.2-3B-Instruct.Q4_K_M.gguf", "system_role": True, "repeat_penalty": 1.0},
    "llama3.2-finetuned": {"ollama_name": "llama3.2-finetuned", "gguf_repo": "Makaareeem/llama3.2-finetuned-gguf",
        "gguf_filename": "Llama-3.2-3B-Instruct.Q4_K_M.gguf", "system_role": True, "repeat_penalty": 1.0},
    "gemma2-base": {"ollama_name": "gemma2-base", "gguf_repo": "Makaareeem/gemma2-base-gguf",
        "gguf_filename": "gemma-2-2b-it.Q4_K_M.gguf", "system_role": False, "repeat_penalty": 1.0},
    "gemma2-finetuned": {"ollama_name": "gemma2-finetuned", "gguf_repo": "Makaareeem/gemma2-finetuned-gguf",
        "gguf_filename": "gemma-2-2b-it.Q4_K_M.gguf", "system_role": False, "repeat_penalty": 1.0},
}

# K5 (LLM komersial) - DeepSeek, closed-source, arsitektur berbeda dari kandidat SLM
COMMERCIAL_MODEL = {"provider": "deepseek", "model": "deepseek-v4-flash",
                     "base_url": "https://api.deepseek.com", "api_key_secret": "API_Deepseek"}

RAGAS_JUDGE_MODEL = "gpt-4o-mini"  
 
SCENARIOS = [
    {"architecture": arch, "condition": condition, "model_key": f"{arch}-{state}", "use_rag": rag}
    for arch in ["gemma2", "llama3.2"]
    for condition, state, rag in [("K1", "base", False), ("K2", "base", True),
                                  ("K3", "finetuned", False), ("K4", "finetuned", True)]
]
SCENARIOS.append({"architecture": "deepseek", "condition": "K5",
                   "model_key": "deepseek-flash", "use_rag": True})

## 4. Bekukan revisi HF dan unduh artefak

In [5]:
from huggingface_hub import HfApi, hf_hub_download

api = HfApi(token=HF_TOKEN)
previous = json.loads((OUT / "manifest.json").read_text(encoding="utf-8")) if (OUT / "manifest.json").exists() else {}
saved = previous.get("asset_revisions", {})

ASSET_REVISIONS = {
    "dataset": api.repo_info(DATASET_REPO, repo_type="dataset", revision=saved.get("dataset", DATASET_REVISION)).sha,
    "kb": api.repo_info(HF_KB_REPO, repo_type="dataset", revision=saved.get("kb", KB_REVISION)).sha,
    "embedding": api.repo_info(EMBEDDING_MODEL_ID, revision=saved.get("embedding", EMBEDDING_REVISION)).sha,
    "reranker": api.repo_info(RERANKER_MODEL_ID, revision=saved.get("reranker", RERANKER_REVISION)).sha,
}
for key, model in MODEL_REGISTRY.items():
    ASSET_REVISIONS[key] = api.repo_info(model["gguf_repo"], revision=saved.get(key, "main")).sha

TEST_PATH = Path(hf_hub_download(DATASET_REPO, f"{DATA_SPLIT}.csv", repo_type="dataset",
    revision=ASSET_REVISIONS["dataset"], token=HF_TOKEN))
for name in ["chunk_metadata.jsonl", "faiss_index.bin"]:
    hf_hub_download(HF_KB_REPO, name, repo_type="dataset", revision=ASSET_REVISIONS["kb"],
                    token=HF_TOKEN, local_dir=KB_DIR)
print("Revisi HF dibekukan:", DATA_SPLIT)

test.csv: 0.00B [00:00, ?B/s]

chunk_metadata.jsonl:   0%|          | 0.00/45.7M [00:00<?, ?B/s]

faiss_index.bin:   0%|          | 0.00/69.4M [00:00<?, ?B/s]

Revisi HF dibekukan: test


In [6]:
import pandas as pd
 
test = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
required = {"record_id", "instruction", "split", "context_chunk_ids", "subtask_type"}
assert required <= set(test), required - set(test)
assert test.record_id.is_unique and test.record_id.str.strip().ne("").all()
assert test["split"].eq(DATA_SPLIT).all(), "Kolom split tidak sesuai DATA_SPLIT."
test = test.sort_values("record_id").reset_index(drop=True)
 
if SMOKE_TEST:
    test = test.sample(n=min(SMOKE_TEST_N, len(test)), random_state=SEED).sort_values("record_id")
else:
    # stratified sampling per subtask_type, proporsional, 5% dari populasi
    test = (test.groupby("subtask_type", group_keys=False)
                .apply(lambda g: g.sample(frac=SAMPLE_FRACTION, random_state=SEED))
                .sort_values("record_id").reset_index(drop=True))
test = test.copy()
 
selected_path = OUT / "test_selected.csv"
if selected_path.exists():
    prior = pd.read_csv(selected_path, dtype=str, keep_default_na=False)
    assert prior.to_dict("records") == test.to_dict("records"), "Sampel berubah; gunakan RUN_NAME baru."
else:
    test.to_csv(selected_path, index=False)
print("SMOKE_TEST" if SMOKE_TEST else f"SAMPLE {SAMPLE_FRACTION:.0%}", "-", len(test), "pertanyaan")
print(test.subtask_type.value_counts())

SAMPLE 5% - 89 pertanyaan
subtask_type
source_navigation           14
definition_clarification    14
methodology_explanation     12
factual_retrieval           12
multi_source_synthesis       8
causal_reasoning             8
trend_interpretation         8
comparative_analysis         8
document_overview            5
Name: count, dtype: int64


/tmp/ipykernel_16/2655617173.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=SAMPLE_FRACTION, random_state=SEED))


## 5. Penentuan gold chunk (untuk Hit@K, Recall@K, MRR)

Sesuai Bab 3: `context_chunk_ids` dipersempit ke chunk yang memuat kutipan `context_spans` verbatim (setelah normalisasi spasi). Fallback ke seluruh `context_chunk_ids` bila tidak ada chunk yang cocok utuh. Dicatat juga kecocokan level section sebagai ukuran relevansi longgar.

In [7]:
import re
 
CONTEXT_SPAN_SEPARATOR = re.compile(r"\n+\s*-{3,}\s*\n+")
 
with open(KB_DIR / "chunk_metadata.jsonl", "r", encoding="utf-8") as f:
    CHUNKS_BY_ID = {json.loads(line)["chunk_id"]: json.loads(line) for line in f}
 
# parse kolom list yang tersimpan sebagai string (json atau python-repr) - dipakai untuk context_chunk_ids
def parse_list_field(raw):
    if not raw or not str(raw).strip():
        return []
    try:
        value = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        try:
            value = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            return [raw]
    return value if isinstance(value, list) else [value]
 
# context_joined: beberapa kutipan verbatim digabung satu string, dipisah "\n\n---\n\n"
def split_context_spans(raw):
    if not raw or not str(raw).strip():
        return []
    return [s.strip() for s in CONTEXT_SPAN_SEPARATOR.split(str(raw)) if s.strip()]
 
# normalisasi spasi untuk pencocokan verbatim
def normalize_spaces(text):
    return " ".join(str(text).split())
 
# tentukan gold chunk satu record: chunk yang memuat >=1 kutipan verbatim, fallback ke seluruh context_chunk_ids
def determine_gold_chunks(record):
    chunk_ids = parse_list_field(record.get("context_chunk_ids", ""))
    spans_norm = [normalize_spaces(s) for s in split_context_spans(record.get("context_joined", ""))]
 
    matched = []
    for cid in chunk_ids:
        chunk = CHUNKS_BY_ID.get(cid)
        if chunk is None:
            continue
        chunk_text_norm = normalize_spaces(chunk["text"])
        if any(span in chunk_text_norm for span in spans_norm):
            matched.append(cid)
 
    gold_ids = matched if matched else chunk_ids
    fallback_used = not matched and bool(chunk_ids)
 
    section_key = None
    first_chunk = CHUNKS_BY_ID.get(chunk_ids[0]) if chunk_ids else None
    if first_chunk:
        section_key = (first_chunk.get("document_title"), first_chunk.get("section_id"))
 
    return {"record_id": record["record_id"], "gold_chunk_ids": gold_ids,
            "gold_section": section_key, "fallback_used": fallback_used,
            "n_spans": len(spans_norm), "n_context_chunks": len(chunk_ids)}
 
gold = pd.DataFrame([determine_gold_chunks(row) for _, row in test.iterrows()])
gold.to_json(OUT / "gold_chunks.jsonl", orient="records", lines=True, force_ascii=False)
print("Gold chunk fallback dipakai pada", int(gold.fallback_used.sum()), "dari", len(gold), "pertanyaan")
print("Rata-rata gold chunk per pertanyaan:", gold.gold_chunk_ids.map(len).mean().round(2))

Gold chunk fallback dipakai pada 80 dari 89 pertanyaan
Rata-rata gold chunk per pertanyaan: 1.0


## 6. Simpan manifest awal

In [8]:
PIPELINE_HASH = "31e573a7c9d5448ee96dc00b693b21a711c636075e2adfdc33968bf46acafc11"
 
manifest = {
    "run_name": RUN_NAME, "data_split": DATA_SPLIT, "smoke_test": SMOKE_TEST,
    "smoke_test_n": SMOKE_TEST_N if SMOKE_TEST else None,
    "sample_fraction": None if SMOKE_TEST else SAMPLE_FRACTION, "seed": SEED,
    "asset_revisions": ASSET_REVISIONS, "pipeline_hash": PIPELINE_HASH,
    "model_registry": MODEL_REGISTRY, "scenarios": SCENARIOS,
    "system_prompt_hash": digest(SYSTEM_PROMPT),
    "system_prompt_matches_training": False,
    "system_prompt_note": ("SYSTEM_PROMPT evaluasi memakai versi produksi terbaru (pasca "
        "perbaikan sitasi/halusinasi), berbeda dari system prompt saat fine-tuning. Keputusan "
        "disengaja: versi training menghasilkan performa lebih rendah pada pengujian awal. "
        "Didokumentasikan sebagai limitasi Bab 5, bukan train-inference skew yang tidak disadari."),
    "gold_chunk_fallback_rate": float(gold.fallback_used.mean()),
    "pilot": SMOKE_TEST,
}
write_json(OUT / "manifest.json", manifest)
print("manifest.json disimpan di", OUT)

manifest.json disimpan di /kaggle/working/rag_results/Penuh
